In [ ]:
# 可视化PCR数据
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# 持仓量PCR
axes[0].plot(pcr_df['日期'], pcr_df['持仓量PCR'], label='持仓量PCR', color='blue', alpha=0.7)
axes[0].axhline(y=1, color='red', linestyle='--', label='PCR=1(中性)')
axes[0].set_ylabel('持仓量PCR', fontproperties=font_prop)
axes[0].set_title('50ETF期权PCR走势', fontproperties=font_prop)
axes[0].legend(prop=font_prop)
axes[0].grid(True, alpha=0.3)

# 成交量PCR
axes[1].plot(pcr_df['日期'], pcr_df['成交量PCR'], label='成交量PCR', color='green', alpha=0.7)
axes[1].axhline(y=1, color='red', linestyle='--', label='PCR=1(中性)')
axes[1].set_ylabel('成交量PCR', fontproperties=font_prop)
axes[1].set_xlabel('日期', fontproperties=font_prop)
axes[1].legend(prop=font_prop)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# PCR解读：
# PCR > 1: 认沽持仓/成交量大于认购，市场情绪偏悲观（避险需求强）
# PCR < 1: 认购持仓/成交量大于认沽，市场情绪偏乐观
# PCR极高时可能是市场底部信号，PCR极低时可能是市场顶部信号

# 1.数据预处理
## 1.1获取国证A指

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import akshare as ak

d:\software\Anaconda\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
d:\software\Anaconda\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [92]:
index_hist_cni_df = ak.index_hist_cni(symbol="399317", start_date="20100104", end_date="20251231")
index_hist_cni_df.to_csv('index_hist_cni_399317.csv', index=False)
index_hist_cni_df.head()

,日期,开盘价,最高价,最低价,收盘价,涨跌幅,成交量,成交额
0,2010-01-04,3311.7279,3316.1241,3276.2273,3276.2273,-0.0060,15678.34,2069.70
1,2010-01-05,3283.1592,3309.1449,3245.8613,3305.7476,0.0090,18735.01,2549.08
2,2010-01-06,3301.4769,3327.4824,3288.7061,3289.3477,-0.0050,18338.14,2503.23
3,2010-01-07,3287.4944,3299.9601,3208.0001,3224.5501,-0.0197,18508.14,2480.13
4,2010-01-08,3210.6486,3244.4500,3189.2038,3243.8944,0.0060,14506.98,1924.86


# 2.构造择时因子
## 2.1计算指数收益率

In [93]:
df = pd.read_csv('index_hist_cni_399317.csv')
df['ret'] = df['收盘价'].pct_change()
df.dropna(inplace=True)
df.head()

,日期,开盘价,最高价,最低价,收盘价,涨跌幅,成交量,成交额,ret
1,2010-01-05,3283.1592,3309.1449,3245.8613,3305.7476,0.0090,18735.01,2549.08,0.009010
2,2010-01-06,3301.4769,3327.4824,3288.7061,3289.3477,-0.0050,18338.14,2503.23,-0.004961
3,2010-01-07,3287.4944,3299.9601,3208.0001,3224.5501,-0.0197,18508.14,2480.13,-0.019699
4,2010-01-08,3210.6486,3244.4500,3189.2038,3243.8944,0.0060,14506.98,1924.86,0.005999
5,2010-01-11,3319.9931,3324.2167,3237.4737,3253.6659,0.0030,19480.67,2673.78,0.003012


## 2.2 动量因子（MOM）
$$ MOM_{t}=\frac{P_{t}}{P_{t-N}}-1 $$

In [94]:
def calc_mom(price, window):
    return price / price.shift(window) - 1

## 2.3 均线偏离（SMA）
$$ SMA\_signal_{t} = \frac{P_{t}}{MA_{N}} - 1 $$

In [95]:
def calc_sma(price, window):
    return price / price.rolling(window).mean() - 1


In [96]:
# 择时信号
def factor_to_signal(factor):
    return (factor > 0).astype(int)


# #正向 vs 反向
# signal_pos = signal
# signal_neg = 1 - signal

#计算择时策略收益 & 净值
def calc_strategy_ret(ret, signal):
    return ret * signal


#计算夏普比率
def sharpe_ratio(ret, freq=252):
    if ret.std() == 0:
        return np.nan
    return np.sqrt(freq) * ret.mean() / ret.std()


In [97]:
# 参数区间
windows = [5, 10, 20, 40, 60]

sharpe_dict = {
    'MOM_pos': [],
    'MOM_neg': [],
}

for w in windows:
    factor = calc_mom(df['收盘价'], w)
    signal = factor_to_signal(factor)
    
    signal_pos = signal
    signal_neg = 1 - signal


    ret_pos = calc_strategy_ret(df['ret'], signal)
    ret_neg = calc_strategy_ret(df['ret'], 1 - signal)
    
    sharpe_dict['MOM_pos'].append(sharpe_ratio(ret_pos))
    sharpe_dict['MOM_neg'].append(sharpe_ratio(ret_neg))



In [98]:
self_sharpe = sharpe_ratio(df['ret'])
self_sharpe

0.32534846403004425

In [ ]:
# 设置中文字体
from matplotlib import font_manager
font_path = "C:/Windows/Fonts/simhei.ttf"  # 替换为系统中实际的中文字体路径
font_prop = font_manager.FontProperties(fname=font_path)

# 绘图
box_data = [
    sharpe_dict['MOM_pos'],
    sharpe_dict['MOM_neg']
 ]

labels = ['MOM_pos', 'MOM_neg']

plt.figure(figsize=(8, 5))

plt.boxplot(box_data, labels=labels, showfliers=True)

plt.axhline(self_sharpe, color='red', linestyle='--', label='SELF')

plt.ylabel('夏普比率', fontproperties=font_prop)
plt.title('择时因子有效性检验', fontproperties=font_prop)
plt.legend(prop=font_prop)

plt.show()

KeyboardInterrupt: 

KeyboardInterrupt: 

# 3 成分股数据

In [3]:
cons = pd.read_excel('399317_cons.xls')
cons.head()

,日期,样本代码,样本简称,所属行业,权重（%）
0,2026-02-06,300750,宁德时代,工业,1.77
1,2026-02-06,600519,贵州茅台,主要消费,1.6
2,2026-02-06,601318,中国平安,金融,1.26
3,2026-02-06,601899,紫金矿业,原材料,1.04
4,2026-02-06,600036,招商银行,金融,0.9


In [4]:
cons = pd.read_csv("399317_cons_prefixed.csv")

# 假设列名叫 symbol
stock_list = cons['code_prefixed'].tolist()

a = stock_list[:5]

In [5]:
def fetch_one_stock(symbol, start_date, end_date, retries=3, pause=0.5):
    """Fetch one stock with simple retry logic."""
    import time
    for attempt in range(retries):
        try:
            df = ak.stock_zh_a_hist_tx(
                symbol=symbol,
                start_date=start_date,
                end_date=end_date,
                adjust="hfq"  # 明确写，防止以后接口默认改
            )
            if df is None or df.empty:
                return None

            df = df.copy()
            df['stock'] = symbol
            df['date'] = pd.to_datetime(df['date'])
            return df[['date', 'stock', 'open', 'high', 'low', 'close', 'amount']]
        except Exception as e:
            # 最后一次重试仍失败则抛出，外层调用可捕获并记录
            if attempt < retries - 1:
                time.sleep(pause * (attempt + 1))
                continue
            else:
                raise

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time, traceback, pandas as pd

# Safe wrapper for pool workers (uses the fetch_one_stock defined earlier)
def fetch_one_stock_safe(symbol):
    try:
        return fetch_one_stock(symbol, "20100101", "20251231")
    except Exception as e:
        print(f"{symbol} 获取失败: {e}")
        return None

all_data = []
n = len(stock_list)
# 线程数：在合理范围内，避免对接口造成过大压力（可根据实际情况调整）
max_workers = min(20, max(4, n // 5))

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(fetch_one_stock_safe, s): s for s in stock_list}
    for fut in tqdm(as_completed(futures), total=len(futures)):
        try:
            df_one = fut.result()
            if df_one is not None:
                all_data.append(df_one)
        except Exception as e:
            s = futures.get(fut, 'unknown')
            print(f"{s} 任务失败: {e}")
        # 每抓取 50 个保存一次，防止意外中断导致全部丢失
        if len(all_data) % 50 == 0 and len(all_data) > 0:
            try:
                pd.concat(all_data).to_parquet('all_data_partial.parquet', index=False)
            except Exception as e:
                print('保存部分结果失败:', e)

# 最终合并并保存
if all_data:
    all_data_df = pd.concat(all_data, ignore_index=True)
    try:
        all_data_df.to_parquet('all_data.parquet', index=False)
    except Exception as e:
        print('保存最终结果失败:', e)
else:
    all_data_df = pd.DataFrame()

# 替换原来的列表为最终 DataFrame，供后续使用
all_data = all_data_df


In [ ]:
# 获取期权持仓量PCR数据
# PCR = Put/Call Ratio = 认沽持仓量 / 认购持仓量

def fetch_50etf_pcr(start_date="20200101", end_date="20251231"):
    """
    获取50ETF期权持仓量PCR数据
    使用akshare的option_daily_stats_sse接口
    """
    from datetime import datetime, timedelta
    import time
    
    # 生成交易日列表
    date_list = []
    current = datetime.strptime(start_date, "%Y%m%d")
    end = datetime.strptime(end_date, "%Y%m%d")
    while current <= end:
        if current.weekday() < 5:  # 跳过周末
            date_list.append(current.strftime('%Y%m%d'))
        current += timedelta(days=1)
    
    pcr_data = []
    failed_dates = []
    
    for date_str in tqdm(date_list, desc="获取50ETF期权PCR数据"):
        try:
            # 获取当日统计数据
            df = ak.option_daily_stats_sse(date=date_str)
            
            if df is not None and not df.empty:
                # 筛选50ETF相关合约（使用正确的列名'合约标的名称'）
                df_50etf = df[df['合约标的名称'].str.contains('50ETF', na=False)].copy()
                
                if not df_50etf.empty:
                    # 提取关键字段
                    row = {
                        '日期': pd.to_datetime(date_str),
                        '合约标的名称': df_50etf['合约标的名称'].values[0],
                        '未平仓认购合约数': df_50etf['未平仓认购合约数'].values[0],
                        '未平仓认沽合约数': df_50etf['未平仓认沽合约数'].values[0],
                        '未平仓合约总数': df_50etf['未平仓合约总数'].values[0],
                        '认沽成交量': df_50etf['认沽成交量'].values[0],
                        '认购成交量': df_50etf['认购成交量'].values[0],
                        # 持仓量PCR = 未平仓认沽合约数 / 未平仓认购合约数
                        '持仓量PCR': df_50etf['未平仓认沽合约数'].values[0] / df_50etf['未平仓认购合约数'].values[0],
                        # 成交量PCR = 认沽成交量 / 认购成交量
                        '成交量PCR': df_50etf['认沽成交量'].values[0] / df_50etf['认购成交量'].values[0],
                    }
                    pcr_data.append(row)
                    
        except Exception as e:
            failed_dates.append(date_str)
            # print(f"获取 {date_str} 数据失败: {e}")
        
        # 适当延时，避免请求过快
        time.sleep(0.1)
    
    if failed_dates:
        print(f"获取失败的日期数: {len(failed_dates)}")
    
    if pcr_data:
        pcr_df = pd.DataFrame(pcr_data)
        return pcr_df
    else:
        return pd.DataFrame()

# 获取数据（可以调整日期范围）
pcr_df = fetch_50etf_pcr(start_date="20200101", end_date="20251231")
print(f"共获取 {len(pcr_df)} 条PCR数据")
pcr_df.head()

In [ ]:
# 保存PCR数据到CSV
pcr_df.to_csv('50etf_pcr_data.csv', index=False, encoding='utf-8-sig')
print("PCR数据已保存到: 50etf_pcr_data.csv")

# 查看数据统计
print("\n=== PCR数据统计 ===")
print(pcr_df[['持仓量PCR', '成交量PCR']].describe())